# 나비야 — 한국어 웨이크워드 학습 (livekit-wakeword)

Colab(GPU) 전용. openWakeWord 호환 `.tflite` 모델을 굽는다 → 로컬 `secrets/navi_ko.tflite` → 우리 어댑터.

설계·배경: `docs/research/d7_wakeword_ko.md`

**2단계 전략:** 1차는 작은 config로 파이프라인·발음·tflite 로드를 빠르게 검증(30분~1h), 통과 후 큰 config로 본학습.

> ⚠️ 아래 명령·config 필드는 livekit README 기준. 버전에 따라 다를 수 있으니 **1-c 셀의 `--help`로 실물 확인** 후 진행.

## 0. 런타임
**런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 선택 후 아래 실행.

In [ ]:
!nvidia-smi

## 1. 시스템 의존성 + 패키지

In [ ]:
!apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox portaudio19-dev

In [ ]:
# voxcpm = 한국어 TTS, tflite = openWakeWord 호환 export
!pip install -q "livekit-wakeword[train,eval,export,voxcpm,tflite]"

### 1-c. 실물 확인 (중요)
서브커맨드·옵션·config 스키마가 아래 가정과 맞는지 먼저 확인.

In [ ]:
!livekit-wakeword --help

## 2. config 작성 — 1차는 작게(스모크)
`tts_backend: voxcpm`(한국어), `model_type: dnn`(openWakeWord 호환 export 전제).

In [ ]:
import os; os.makedirs('configs', exist_ok=True)

In [ ]:
%%writefile configs/navi_ko.yaml
model_name: navi_ko
target_phrases:
  - "나비야"
tts_backend: voxcpm        # 빼면 영어 Piper로 떨어짐
n_samples: 2000            # 1차 스모크. 검증되면 10000+
model:
  model_type: dnn          # openWakeWord 호환(conv_attention 아님)
  model_size: small
steps: 10000               # 1차. 본학습은 50000

## 3. 발음부터 확인 (generate만)
본학습 전에 "나비야" 합성이 한국어로 제대로 나오는지 귀로 확인. 깨지면 여기서 잡는다.

> VoxCPM2 모델 다운로드에서 막히면 HuggingFace 로그인 필요할 수 있음: `!huggingface-cli login`

In [ ]:
!livekit-wakeword generate configs/navi_ko.yaml

In [ ]:
import glob, IPython.display as ipd
clips = sorted(glob.glob('**/navi_ko*/**/*.wav', recursive=True))[:3]
print(clips or '클립 경로를 못 찾음 — generate 출력 폴더 확인')
for c in clips:
    print(c); ipd.display(ipd.Audio(c))

## 4. 학습 + export
`run` = generate→augment→train 일괄. T4에서 1차(2000/10000)는 수십 분~1h. 첫 실행은 VoxCPM2 weight 다운로드 포함.

In [ ]:
!livekit-wakeword run configs/navi_ko.yaml

In [ ]:
!livekit-wakeword export configs/navi_ko.yaml --format tflite

## 5. 다운로드 → 로컬 `secrets/navi_ko.tflite`

In [ ]:
import glob
from google.colab import files
hits = glob.glob('**/navi_ko*.tflite', recursive=True)
print(hits or 'tflite를 못 찾음 — export 출력 경로 확인')
if hits:
    files.download(hits[0])

## 6. 검증 통과 후 본학습
1차에서 **① 발음 OK ② tflite 생성 ③ (로컬) 어댑터 로드·감지 OK** 면 → config를 키워 2~5절 재실행:
- `n_samples: 10000`~`50000`, `steps: 50000`

## 팁
- **Colab 끊김:** 무료 T4는 유휴 끊김·~12h 제한. 긴 학습은 Drive 마운트로 산출물 보존, 또는 `generate`/`augment`/`train` 스테이지 분리 저장.
- **이어받기:** `secrets/navi_ko.tflite` 생기면 어댑터 `inference_framework` 인자화 + config `model_path` + `owww_mic.py` recall 실측은 데몬 쪽에서.